<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module340/Lab7.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 7 — H₂ Bond-Length Curve, Noise, and Optional Hardware
**Quantum Optimization and Simulation — VQE Laboratory Series**

Connect VQE's electronic inner loop to geometry optimization.

**Suggested use:** 10–15 minute instructor demonstration followed by approximately one hour of independent work.

**Notebook style:** Most code is supplied. Complete the small items marked **YOUR TURN** and answer the reflection questions.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Distinguish electronic optimization at fixed bond length from geometry optimization.
- Construct a simple H₂ potential-energy curve from supplied Hamiltonians.
- Compare exact, shot-based, and noisy estimates.
- Optionally transpile/run one point on IBM hardware.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

In [ ]:
from scipy.optimize import minimize
from qiskit_aer.noise import NoiseModel, depolarizing_error

## Supplied teaching data

The following coefficient sets are a compact teaching data set. They illustrate the VQE/geometry workflow; the complete professional workflow would regenerate molecular integrals at each distance using a chemistry driver.

In [ ]:
# R in Angstrom, followed by c0,c1,c2,c3,c4
h2_data = {
    0.50: (-0.72,  0.44, -0.44, -0.02, 0.16),
    0.60: (-0.92,  0.42, -0.42, -0.015, 0.17),
    0.70: (-1.04,  0.40, -0.40, -0.012, 0.18),
    0.74: (-1.0523732458, 0.3979374248, -0.3979374248, -0.0112801043, 0.1809311998),
    0.80: (-1.02,  0.38, -0.38, -0.010, 0.18),
    1.00: (-0.90,  0.32, -0.32, -0.008, 0.17),
    1.40: (-0.72,  0.20, -0.20, -0.004, 0.14),
}

In [ ]:
def hamiltonian_from_coeffs(coeffs):
    c0,c1,c2,c3,c4 = coeffs
    return SparsePauliOp.from_list([
        ("II", c0),
        ("IZ", c1),
        ("ZI", c2),
        ("ZZ", c3),
        ("XX", c4),
        ("YY", c4),
    ])

exact_curve = {}
for R, coeffs in h2_data.items():
    H = hamiltonian_from_coeffs(coeffs)
    exact_curve[R] = np.linalg.eigvalsh(H.to_matrix())[0]

plt.plot(list(exact_curve.keys()), list(exact_curve.values()), "o-")
plt.xlabel("Bond distance R (Angstrom)")
plt.ylabel("Ground-state energy (Hartree)")
plt.title("Teaching H2 potential-energy curve")
plt.show()

## Part B — Add sampling noise at one distance

In [ ]:
from qiskit.circuit.library import XXPlusYYGate

R = 0.74
coeffs = h2_data[R]
c0,c1,c2,c3,c4 = coeffs

def ansatz(theta):
    qc = QuantumCircuit(2)
    qc.x(0)
    # Phase-adjusted XX+YY gate: real Givens-style mixing.
    qc.append(XXPlusYYGate(2*theta, beta=np.pi/2), [0, 1])
    return qc

def parity_expectation(counts, qubits):
    total = sum(counts.values())
    result = 0.0
    for bits, count in counts.items():
        value = 1
        for q in qubits:
            bit = int(bits[-1-q])
            value *= 1 if bit == 0 else -1
        result += value * count/total
    return result

def sampled_energy(theta_array, noise_model=None):
    theta = float(np.atleast_1d(theta_array)[0])

    values = {}
    for basis in ["Z","X","Y"]:
        qc = ansatz(theta)
        if basis == "X":
            qc.h([0,1])
        elif basis == "Y":
            qc.sdg([0,1])
            qc.h([0,1])
        qc.measure_all()
        counts = run_counts(qc, shots=2048, noise_model=noise_model)
        values[basis] = counts

    z0 = parity_expectation(values["Z"], [0])
    z1 = parity_expectation(values["Z"], [1])
    zz = parity_expectation(values["Z"], [0,1])
    xx = parity_expectation(values["X"], [0,1])
    yy = parity_expectation(values["Y"], [0,1])

    return c0+c1*z0+c2*z1+c3*zz+c4*(xx+yy)

In [ ]:
ideal_result = minimize(sampled_energy, x0=[0.0], method="COBYLA",
                        options={"maxiter":35})

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.003, 1), ["x","h","sdg","rz","rx"]
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.02, 2), ["cx"]
)

noisy_result = minimize(
    lambda x: sampled_energy(x, noise_model=noise_model),
    x0=[0.0],
    method="COBYLA",
    options={"maxiter":35}
)

print("Exact:", exact_curve[R])
print("Shot-based ideal:", ideal_result.fun)
print("Noisy:", noisy_result.fun)

## Part C — Optional real hardware

Real hardware requires an IBM Quantum account. Use one optimized parameter and estimate the three basis groups. The exact API and available devices can change; therefore this section is intentionally optional and may need a small update when the course runs.

In [ ]:
# OPTIONAL SKELETON
#
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
# from qiskit.transpiler import generate_preset_pass_manager
#
# service = QiskitRuntimeService()
# backend = service.least_busy(min_num_qubits=2, operational=True, simulator=False)
# pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
#
# theta_hw = ideal_result.x[0]
# circuits = []
# for basis in ["Z","X","Y"]:
#     qc = ansatz(theta_hw)
#     if basis == "X":
#         qc.h([0,1])
#     elif basis == "Y":
#         qc.sdg([0,1]); qc.h([0,1])
#     qc.measure_all()
#     circuits.append(pm.run(qc))
#
# sampler = SamplerV2(mode=backend)
# job = sampler.run(circuits, shots=4096)
# print(job.job_id())

### YOUR TURN
From `exact_curve`, identify the bond distance with the lowest energy.

## Reflection
Explain the difference between:
- the inner VQE loop at fixed \(R\), and
- the outer geometry loop over \(R\).

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    ```python
    best_R = min(exact_curve, key=exact_curve.get)
    print(best_R, exact_curve[best_R])
    ```

    The inner loop changes circuit parameters to find the lowest electronic energy for one fixed molecular geometry. The outer loop changes the molecular geometry itself and repeats the electronic calculation.

</details>